# S&P 500 Returns Prediction - Inference Server

This notebook implements the Kaggle Evaluation API for predicting S&P 500 daily returns.

## Setup

Make sure to:
1. Add your trained model file to the notebook (as a dataset or file)
2. Ensure all dependencies are installed
3. The model should be in `models/model_lightgbm.pkl` (or update the path below)


## Install Dependencies


In [ ]:
# Install required packages if not already available
!pip install -q polars grpcio protobuf


## Import Libraries


In [ ]:
import os
import sys
from pathlib import Path

import pandas as pd
import polars as pl
import numpy as np
import joblib

# Import Kaggle evaluation framework
import kaggle_evaluation.default_inference_server

# Note: You'll need to include your src/ directory code here
# For Kaggle, you can either:
# 1. Upload src/ as a dataset and add it to the notebook
# 2. Copy the necessary functions directly into this notebook
# 3. Use the code from your local src/ directory


## Include Source Code

If you uploaded `src/` as a dataset, uncomment and run this cell:


In [ ]:
# Add src to path if uploaded as dataset
# sys.path.insert(0, '/kaggle/input/your-src-dataset/src')

# Or if src is in the same directory as this notebook:
# sys.path.insert(0, str(Path.cwd() / 'src'))


## Define Helper Functions

These functions are from `src/data_loader.py` and `src/models.py`.
Include them here or import from your src/ directory.


In [ ]:
# Feature preparation functions
def get_feature_columns(df):
    """Extract feature column names from dataframe."""
    exclude_cols = [
        'date_id',
        'forward_returns',
        'risk_free_rate',
        'market_forward_excess_returns',
        'is_scored',
        'lagged_forward_returns',
        'lagged_risk_free_rate',
        'lagged_market_forward_excess_returns'
    ]
    feature_cols = [col for col in df.columns if col not in exclude_cols]
    return feature_cols


def prepare_features(df, feature_cols=None):
    """Prepare features for modeling."""
    if feature_cols is None:
        feature_cols = get_feature_columns(df)
    
    X = df[feature_cols].copy()
    
    # Handle missing values - forward fill then backward fill
    X = X.ffill().bfill()
    
    # Fill any remaining NaN with 0
    X = X.fillna(0)
    
    return X


In [ ]:
# Model loading functions
def load_model_from_file(model_path, model_type='lightgbm'):
    """Load a trained model from file."""
    try:
        import lightgbm as lgb
        import xgboost as xgb
        from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
    except ImportError as e:
        print(f"Warning: {e}")
    
    data = joblib.load(model_path)
    
    model = data['model']
    feature_cols = data['feature_cols']
    
    return model, feature_cols


## Global Variables for Model

These will be initialized on the first predict call.


In [ ]:
# Global model variables
_model = None
_feature_cols = None
_model_loaded = False


## Model Loading Function


In [ ]:
def load_model(model_path=None, model_type='lightgbm'):    """    Load the trained model. This is called lazily on the first predict call    to avoid exceeding the 15-minute startup time limit.    """    global _model, _feature_cols, _model_loaded        if _model_loaded:        return  # Model already loaded        # Determine model path if not provided    if model_path is None:        # Try different possible locations        model_types = ['lightgbm', 'ensemble', 'xgboost', 'rf', 'gbm']                for mt in model_types:            for base_path in [                Path('/kaggle/input/sp500-lightgbm-mode'),  # Direct file in dataset root                Path('/kaggle/input/sp500-lightgbm-mode/models'),  # File in models/ subfolder                Path('/kaggle/working/models'),                Path('models'),                Path.cwd() / 'models'            ]:                potential_path = base_path / f'model_{mt}.pkl'                if potential_path.exists():                    model_path = str(potential_path)                    model_type = mt                    print(f"Found model at: {model_path}")                    break            if model_path:                break                if model_path is None:            raise FileNotFoundError(                "No trained model found. Please ensure your model file is uploaded as a dataset or in the working directory."            )        print(f"Loading model from {model_path}...")        # Load model    _model, _feature_cols = load_model_from_file(model_path, model_type)    _model_loaded = True        print(f"Model loaded successfully. Model type: {model_type}")    print(f"Number of features: {len(_feature_cols)}")

## Predict Function

This is the main function that Kaggle's evaluation API will call.


In [ ]:
def predict(test: pl.DataFrame) -> pl.DataFrame:
    """
    Predict function for Kaggle Evaluation API.
    
    This function is called by the evaluation API with batches of test data.
    Each batch (except the first) must be returned within 5 minutes.
    The first call can take longer to load the model.
    
    Args:
        test: Polars DataFrame containing test features for a batch
        
    Returns:
        Polars DataFrame with predictions. Must contain 'date_id' and 'forward_returns' columns.
    """
    global _model, _feature_cols, _model_loaded
    
    # Load model on first call (lazy loading)
    if not _model_loaded:
        model_type = os.getenv('MODEL_TYPE', 'lightgbm')
        load_model(model_type=model_type)
    
    # Convert Polars DataFrame to Pandas for compatibility
    test_pd = test.to_pandas()
    
    # Prepare features
    X_test = prepare_features(test_pd, _feature_cols)
    
    # Make predictions
    predictions = _model.predict(X_test)
    
    # Create result DataFrame
    result = pl.DataFrame({
        'date_id': test_pd['date_id'].values,
        'forward_returns': predictions
    })
    
    return result


## Initialize Inference Server

This sets up the server that will handle requests from Kaggle's evaluation system.


In [ ]:
# Create inference server instance
inference_server = kaggle_evaluation.default_inference_server.DefaultInferenceServer(predict)

print("Inference server initialized successfully!")


## Start Server

**Important**: This cell must be run for the evaluation to work.

When `KAGGLE_IS_COMPETITION_RERUN` is set (during actual evaluation), this will start the server and wait for requests.

For local testing, it will run with a local gateway.




In [ ]:
# Start the inference serverprint("=" * 60)print("Starting Inference Server")print("=" * 60)print(f"Environment: KAGGLE_IS_COMPETITION_RERUN = {os.getenv('KAGGLE_IS_COMPETITION_RERUN')}")print()if os.getenv('KAGGLE_IS_COMPETITION_RERUN'):    # Competition mode: start the server and wait for requests    print("🚀 COMPETITION MODE: Starting server...")    print("Server will wait for requests from Kaggle's evaluation system.")    print("This is normal - the server is running and ready!")    print("=" * 60)    inference_server.serve()else:    # Local testing mode: run with local gateway    print("🧪 TESTING MODE: Running local gateway...")    data_path = os.getenv('DATA_PATH', '/kaggle/input/hull-tactical-market-prediction/')    if not os.path.exists(data_path):        # Try alternative paths        alt_paths = [            '/kaggle/input/hull-tactical-market-prediction/',            '/kaggle/working/data/',            'data/'        ]        for alt_path in alt_paths:            if os.path.exists(alt_path):                data_path = alt_path                break        print(f"Using data path: {data_path}")    inference_server.run_local_gateway((data_path,))